In [2]:
from preprocessing_class import Preprocessing, load_pres
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC,SVC
from nltk.corpus import stopwords
import numpy as np
import string
import matplotlib.pyplot as plt
import pandas as pd
%load_ext autoreload
%autoreload 2

In [3]:
# load dataset
alltxt, alllabs = load_pres("Dataset/corpus.tache1.learn.utf8")
alltxt, alllabs = np.array(alltxt), np.array(alllabs)
# M = -1, C = 1
len(alltxt), len(alllabs), np.unique(alllabs)

(57413, 57413, array([-1,  1]))

In [4]:
tmp_labels = np.where(alllabs == 1, 0, 1)
alllabs = np.where(alllabs == 1, 0, 1)

In [5]:
alllabs

array([0, 0, 0, ..., 0, 0, 0], shape=(57413,))

In [6]:
# train test split here for test in final prediction

X_train, X_test, y_train, y_test = train_test_split(
    alltxt, alllabs, 
    test_size=0.20, 
    stratify=alllabs, 
    random_state=42
)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.20, 
    stratify=y_train, 
    random_state=42
)

len(X_train_sub), len(y_train_sub), len(X_val), len(y_val), len(X_train), len(y_train), len(X_test), len(y_test)

(36744, 36744, 9186, 9186, 45930, 45930, 11483, 11483)

In [25]:
punc = set(string.punctuation + '\n\r\t')
punc.remove("'") # keep the ' for french words 
custom_punctuation = "".join(punc)

prep = Preprocessing(low_case = True,
                rm_punctuation = True,
                rm_number = False,
                word_norm = None, # stem faster
                pos_tagging = False, # to check
                all_capital = True, # keep all capital words as they are
                cap_name = True, # garder les noms en majuscules, for pos tag
                rm_accent = False, 
                lang = "french",
                punct = custom_punctuation, # punctuation can contain -, that would be kept
                urls = False 
                )

In [26]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.ensemble import RandomForestClassifier

final_stopwords_list = stopwords.words('french') # stopwords.words('english') 

def build_vectorizer(name="count", stop_words=None, max_features=None,
                     lowercase=False, strip_accents=None, ngram_range=(1,1), use_idf= True, smooth_idf=True, sublinear_tf=False, 
                     max_df=0.95, min_df=2, preprocessor = lambda x: x):
    if name == "count":
        vectorizer = CountVectorizer(stop_words=stop_words,max_features=max_features,max_df=max_df,min_df=min_df,ngram_range=ngram_range,lowercase=lowercase,
                                     strip_accents=strip_accents, preprocessor=preprocessor)    
        
    elif name == "tfidf":
        vectorizer = TfidfVectorizer(use_idf= use_idf, smooth_idf=smooth_idf, sublinear_tf=sublinear_tf,max_df=max_df, min_df=min_df, max_features=max_features,
                                     stop_words=stop_words, ngram_range=ngram_range,lowercase=lowercase,strip_accents=strip_accents, preprocessor=preprocessor)
    
    return vectorizer

def vectorize_txts(txts,vectorizer):
    X = vectorizer.fit_transform(txts)
    return X

## solver = 'newton-cholesky' to try for logistic regression (n_samples >> n_features * n_classes, especially with one-hot encoded categorical features with rare categories)
def build_model(name="kmeans", n_clusters = 2, max_iter = 1000, penalty='l2', loss='squared_hinge',kernel='rbf',solver='lbfgs'):
    if name == "kmeans":
        return KMeans(n_clusters=n_clusters,max_iter=max_iter) 
    elif name == "rf":
        return RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    elif name == "linear_svm":
        return LinearSVC(class_weight="balanced", penalty=penalty,loss=loss,max_iter=max_iter)
    elif name == "nb":
        return MultinomialNB()
    elif name == "svm":
        return SVC(max_iter=max_iter,kernel=kernel)
    elif name == "logreg":
        return LogisticRegression(class_weight="balanced", solver=solver,max_iter=max_iter) 

Test simple models with counts prep 2

In [27]:
nb = build_model(name="nb")
prep_X_train = [prep.process(t) for t in X_train]
prep_X_test = [prep.process(t) for t in X_test]

In [28]:
vect_count = build_vectorizer(name="count", stop_words=final_stopwords_list, preprocessor=prep)
vect_tf = build_vectorizer(name="tfidf", stop_words=final_stopwords_list, preprocessor=prep)

In [ ]:
# with simple grid search
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
import numpy as np

# --- Pipeline: vectorizer + model ---
pipe = Pipeline([
    ('tfidf', vect_tf),
    ('clf', nb)
])

# --- Tune both vectorizer and model params ---
param_grid = {
    'clf__alpha': [0.1, 1],
}

scoring = {
    'f1': 'f1_macro',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'roc_auc': 'roc_auc_ovr',        # one-vs-rest for multiclass
    'avg_precision': 'average_precision_macro'
}

# --- Grid search with cross-validation on train set only ---
grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    refit='f1', 
    scoring=scoring,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

# --- CV results ---
print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

# --- Evaluate on test set ---
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 2 candidates, totalling 10 fits


In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
cols = ['params'] + [f'mean_test_{m}' for m in scoring]
print(results[cols])

## RNN model

In [44]:
punc = set(string.punctuation + '\n\r\t')
punc.remove("'") # keep the ' for french words 
custom_punctuation = "".join(punc)

# using pos tag first
prep_pos = Preprocessing(low_case = False,
                rm_punctuation = False,
                rm_number = False,
                word_norm = None, # stem faster
                pos_tagging = True, # to check
                all_capital = True, # keep all capital words as they are
                cap_name = True, # garder les noms en majuscules, for pos tag
                rm_accent = False, 
                lang = "french",
                punct = custom_punctuation, # punctuation can contain -, that would be kept
                urls = False 
)


In [ ]:
# prep_X_train = [[p for (t, p) in prep_pos.process(t)[1]] for t in X_train_sub]
# prep_X_val = [[p for (t, p) in prep_pos.process(t)[1]] for t in X_train_val]

In [11]:
prep_X_train = [p for t, p in zip(X_train, [prep_pos.process(t) for t in X_train])]
prep_X_test = [p for t, p in zip(X_test, [prep_pos.process(t) for t in X_test])]

In [21]:
X_train_pos = [[tupl[1] for tupl in t[1]] for t in prep_X_train]
X_test_pos = [[tupl[1] for tupl in t[1]] for t in prep_X_test]

In [23]:
def build_vocab(all_tag_sequences):
    """all_tag_sequences: list of lists, e.g. [['DT','NN','VB'], ['JJ','NN'], ...]"""
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for tags in all_tag_sequences:
        for tag in tags:
            if tag not in vocab:
                vocab[tag] = len(vocab)
    return vocab

# building vocab from train
vocab = build_vocab(X_train_pos)

In [33]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

class POSDataset(Dataset):
    def __init__(self, tag_sequences, labels, vocab):
        self.labels = labels
        self.vocab  = vocab
        # Convert tags → int tensors (variable length, no padding yet)
        self.encoded = [
            torch.tensor([vocab.get(t, vocab["<UNK>"]) for t in seq], dtype=torch.long)
            for seq in tag_sequences
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], self.labels[idx]
    

def collate_fn(batch):
    """
    Called by DataLoader for each batch.
    Pads sequences to the length of the longest one in the batch.
    """
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences])

    # pad_sequence stacks tensors and pads shorter ones with 0 (<PAD>)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.tensor(labels, dtype=torch.long)

    return padded, lengths, labels

dataset_train    = POSDataset(X_train_pos, y_train, vocab)
dataloader_train = DataLoader(dataset_train, batch_size=2, shuffle=True, collate_fn=collate_fn)

dataset_test    = POSDataset(X_test_pos, y_test, vocab)
dataloader_test = DataLoader(dataset_test, batch_size=2, shuffle=True, collate_fn=collate_fn)

In [34]:
class POSClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.3):
        super().__init__()

        # Embedding: tag_id → dense vector. padding_idx=0 keeps <PAD> as zero vector.
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,   # reads sequence left→right AND right→left
            dropout=dropout if num_layers > 1 else 0,
        )

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional

    def forward(self, x, lengths):
        # x: (batch, max_seq_len)  lengths: actual length of each sequence

        emb = self.dropout(self.embedding(x))          # (batch, seq, embed_dim)

        # pack_padded_sequence tells the LSTM to skip PAD tokens entirely
        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hn, _) = self.lstm(packed)
        # hn shape: (num_directions * num_layers, batch, hidden_dim)

        # Take the last layer's forward + backward hidden states
        # hn[-2] = last forward,  hn[-1] = last backward
        hn = torch.cat([hn[-2], hn[-1]], dim=1)        # (batch, hidden_dim * 2)

        return self.classifier(self.dropout(hn))       # (batch, num_classes)


model = POSClassifier(
    vocab_size=len(vocab),
    embed_dim=32,        # small: POS tagsets have ~15–50 tags
    hidden_dim=64,
    num_classes=2,       # change to your number of classes
    num_layers=2,
    dropout=0.3,
)
print(model)

POSClassifier(
  (embedding): Embedding(19, 32, padding_idx=0)
  (lstm): LSTM(32, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [35]:
def compute_class_weights(labels):
    counts  = torch.bincount(torch.tensor(labels)).float()
    weights = 1.0 / counts
    weights = weights / weights.sum()
    return weights

class_weights = compute_class_weights(y_train)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
optimizer     = torch.optim.Adam(model.parameters(), lr=1e-3)

In [36]:
from sklearn.metrics import (
    f1_score, average_precision_score,
    roc_auc_score, precision_score, recall_score
)

def compute_metrics(all_labels, all_preds, all_probs):
    """
    Binary classification metrics.
    all_probs: probability of class 1 only — shape (N,)
    """
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    return {
        "f1":            f1_score(all_labels, all_preds, average="binary"),
        "avg_precision": average_precision_score(all_labels, all_probs),
        "roc_auc":       roc_auc_score(all_labels, all_probs),
        "precision":     precision_score(all_labels, all_preds, zero_division=0),
        "recall":        recall_score(all_labels, all_preds),
    }


In [37]:
def run_epoch(model, loader, optimizer, criterion, training=True):
    model.train() if training else model.eval()

    total_loss = 0
    all_labels = []
    all_preds  = []
    all_probs  = []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for padded, lengths, labels in loader:

            if training:
                optimizer.zero_grad()

            logits = model(padded, lengths)                 # (batch, 2)
            loss   = criterion(logits, labels)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            probs = torch.softmax(logits, dim=1)            # (batch, 2)
            preds = probs.argmax(dim=1)

            total_loss += loss.item()
            all_labels .extend(labels.cpu().numpy())
            all_preds  .extend(preds.cpu().numpy())
            all_probs  .extend(probs[:, 1].detach().cpu().numpy())  # prob of class 1

    metrics        = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = total_loss / len(loader)
    return metrics

In [32]:
best_f1        = 0.0
best_model_state = None

In [38]:
for epoch in range(1):
    train_m = run_epoch(model, dataloader_train, optimizer, criterion, training=True)
    val_m   = run_epoch(model, dataloader_test,  optimizer, criterion, training=False)

    # Save best model by val F1
    if val_m["f1"] > best_f1:
        best_f1          = val_m["f1"]
        best_model_state = model.state_dict().copy()

    print(
        f"Epoch {epoch+1:2d} | "
        f"loss  train {train_m['loss']:.4f}  val {val_m['loss']:.4f} | "
        f"f1    train {train_m['f1']:.3f}  val {val_m['f1']:.3f} | "
        f"auc   train {train_m['roc_auc']:.3f}  val {val_m['roc_auc']:.3f} | "
        f"ap    train {train_m['avg_precision']:.3f}  val {val_m['avg_precision']:.3f} | "
        f"prec  train {train_m['precision']:.3f}  val {val_m['precision']:.3f} | "
        f"rec   train {train_m['recall']:.3f}  val {val_m['recall']:.3f}"
    )

print(f"\nBest val F1: {best_f1:.3f}")

Epoch  1 | loss  train 0.6428  val 0.6191 | f1    train 0.047  val 0.229 | auc   train 0.650  val 0.715 | ap    train 0.243  val 0.316 | prec  train 0.512  val 0.491 | rec   train 0.024  val 0.150

Best val F1: 0.229


In [39]:
for epoch in range(1):
    train_m = run_epoch(model, dataloader_train, optimizer, criterion, training=True)
    val_m   = run_epoch(model, dataloader_test,  optimizer, criterion, training=False)

    # Save best model by val F1
    if val_m["f1"] > best_f1:
        best_f1          = val_m["f1"]
        best_model_state = model.state_dict().copy()

    print(
        f"Epoch {epoch+1:2d} | "
        f"loss  train {train_m['loss']:.4f}  val {val_m['loss']:.4f} | "
        f"f1    train {train_m['f1']:.3f}  val {val_m['f1']:.3f} | "
        f"auc   train {train_m['roc_auc']:.3f}  val {val_m['roc_auc']:.3f} | "
        f"ap    train {train_m['avg_precision']:.3f}  val {val_m['avg_precision']:.3f} | "
        f"prec  train {train_m['precision']:.3f}  val {val_m['precision']:.3f} | "
        f"rec   train {train_m['recall']:.3f}  val {val_m['recall']:.3f}"
    )

print(f"\nBest val F1: {best_f1:.3f}")

Epoch  1 | loss  train 0.6160  val 0.5576 | f1    train 0.160  val 0.154 | auc   train 0.687  val 0.722 | ap    train 0.295  val 0.331 | prec  train 0.522  val 0.575 | rec   train 0.095  val 0.089

Best val F1: 0.229


In [ ]:
for epoch in range(1):
    train_m = run_epoch(model, dataloader_train, optimizer, criterion, training=True)
    val_m   = run_epoch(model, dataloader_test,  optimizer, criterion, training=False)

    # Save best model by val F1
    if val_m["f1"] > best_f1:
        best_f1          = val_m["f1"]
        best_model_state = model.state_dict().copy()

    print(
        f"Epoch {epoch+1:2d} | "
        f"loss  train {train_m['loss']:.4f}  val {val_m['loss']:.4f} | "
        f"f1    train {train_m['f1']:.3f}  val {val_m['f1']:.3f} | "
        f"auc   train {train_m['roc_auc']:.3f}  val {val_m['roc_auc']:.3f} | "
        f"ap    train {train_m['avg_precision']:.3f}  val {val_m['avg_precision']:.3f} | "
        f"prec  train {train_m['precision']:.3f}  val {val_m['precision']:.3f} | "
        f"rec   train {train_m['recall']:.3f}  val {val_m['recall']:.3f}"
    )

print(f"\nBest val F1: {best_f1:.3f}")

In [ ]:
model.load_state_dict(best_model_state)

def predict(model, tag_sequence, vocab, threshold=0.5):
    """
    threshold: lower it (e.g. 0.3) to catch more positives if recall matters more
    """
    model.eval()
    ids     = torch.tensor([[vocab.get(t, vocab["<UNK>"]) for t in tag_sequence]])
    lengths = torch.tensor([len(tag_sequence)])

    with torch.no_grad():
        logits   = model(ids, lengths)
        prob_pos = torch.softmax(logits, dim=1)[0, 1].item()

    label = 1 if prob_pos >= threshold else 0
    return {"label": label, "prob": round(prob_pos, 4)}